# Part 3: The Machine Learning Workflow — applied to Sarah's sentiment project
**⏱ This section takes approximately 30 minutes.**

---

## Scenario: Thursday afternoon — the report is due tomorrow

Sarah has a working sentiment model. The numbers are ready. But before she walks into Priya's Friday meeting, she wants to make sure she can answer **every** question Priya and Marcus (finance) are likely to ask.

So she steps back and asks herself: *"Did I actually follow a proper process? Or did I get lucky?"*

This notebook walks through the **7-step ML workflow** and checks Sarah's sentiment project against each step. No new modelling — just an honest retrospective.

**By the end of this section you will:**

- Know the 7 steps every ML project follows, in order
- Recognise that framing (Step 1) is where most projects succeed or fail
- See how Sarah's sentiment work maps onto the workflow — including where she *skipped* steps
- Be able to apply the same workflow to a brand-new business problem

> *Sarah Chen · Customer Experience Analyst · NorthStar Retail · January 2023 (Thursday afternoon).*
>
> The 10,000 reviews are labelled. The topic groupings are ready. Priya is expecting the deck tomorrow at 9am. Sarah closes her laptop, opens a fresh page in her notebook, and writes at the top: **"Did I do this right?"**

## Setup

In [ ]:
# A few light-weight tools — we do not train anything new in this notebook.
import pandas as pd
from textblob import TextBlob

print("✅ Ready!")

## The 7-step workflow

Every serious ML project follows these steps, in this order:

```
1. Frame the problem          Business question · success metric · who acts on the output
2. Collect the data           Where does it live? Is there enough? Is it labelled?
3. Clean and explore          Missing values · duplicates · obvious bias
4. Train the model            Choose a method and fit it — or use someone else's pre-trained model
5. Evaluate on unseen data    Does it generalise? Does it answer the real question?
6. Deploy                     Make it usable by people or systems (not just a notebook)
7. Monitor                    The world changes; the model decays
```

**Surprising time split on a real project:**

| Phase | % of project time |
|---|---|
| Steps 1–3 (framing, collection, cleaning) | **60–70%** |
| Steps 4–5 (training, evaluation) | **10–20%** |
| Steps 6–7 (deployment, monitoring) | **20%** |

Most beginners expect it the other way round. It isn't.

## Step 1 — Frame the problem (the make-or-break step)

Before any code, Sarah should be able to write down four things in plain English:

1. **The question.** *Which of these 10,000 reviews are positive, which are negative, and what are the negative ones about?*
2. **Who acts on the answer.** Priya, who will present to the leadership team; and Aisha in Customer Service, who will use the negative-theme breakdown to brief her agents next week.
3. **Success metric.** The report lands on Friday; the leadership team gets a clear picture of where unhappiness is concentrated; Aisha can point her team at the top 1–2 pain points.
4. **Cost of being wrong.**
   - *Wrongly labelled as positive* (a negative review sneaks into the positive pile): Aisha's team underestimates a real pain point.
   - *Wrongly labelled as negative*: a small amount of time wasted reading it.

**False reassurance is more costly than false alarm here.** Sarah should tilt toward over-flagging negatives when she's uncertain. Different business, different cost ratio — always ask.

### ⏸️ Sarah's own check-in

Sarah closes her eyes and asks herself: *"If Priya pushes back tomorrow, what would she push back on?"*

**Name two things that are still unclear from the framing above** — questions Sarah should have nailed down on Monday before she touched any data.

### What Sarah realises she should have asked

- **What exactly counts as "negative"?** A 3-star review with mild complaints? Or only 1–2 stars? Different definitions give very different counts.
- **What counts as "about sizing"?** A review that says *"runs small but love the colour"* — is that a sizing complaint, a positive review, or both?
- **Does the 10,000 include repeat customers writing multiple reviews?** If the same frustrated customer appears 5 times, are they counted 5 times?
- **What time period do these cover?** A spike in complaints might reflect one bad shipment, not a pattern.
- **Who makes the final call on the theme buckets** — Sarah or Aisha? Aisha owns the outcome.

**Pushing back on the ask is part of the job. A well-framed question is worth more than a clever model.**

## Step 2 — Collect the data

Sarah's data source was Aisha's USB drive: **10,000 customer reviews** pulled from the online store's review system.

For a sentiment task she needs:

- **Features:** the text of each review.
- **Labels:** a known positive / negative tag for each review — needed only if she's going to *train* her own model.

Because she used a **pre-trained model** (TextBlob, trained by someone else on a large body of text), she didn't need labels for training. She only needs labels for **evaluation** (Step 5), and only on a small sample.

> **What could have gone wrong here?** If the USB drive only contained reviews from customers who had opened a support ticket, the dataset would be systematically skewed negative. Sarah should check this with Aisha. (She did — Aisha confirmed it's a full dump from the review system, not just complaints.)

## Step 3 — Clean and explore

Before running the sentiment model, Sarah spent 20 minutes just *looking* at the reviews.

In [ ]:
# A tiny sample that stands in for what Sarah saw when she opened the file.
reviews_sample = pd.DataFrame({
    "review": [
        "Absolutely love this dress, fits perfectly and the colour is gorgeous!",
        "Runs way too small, had to return. Very disappointed.",
        "Arrived two weeks late. Will not order again.",
        "Great quality fabric, washes well, recommend.",
        "Size chart is misleading — medium is more like small.",
        "",  # blank review — happens in real data
        "ok",  # very short, hard to tell
        "Absolutely love this dress, fits perfectly and the colour is gorgeous!",  # duplicate
    ],
})

print("Total rows:", len(reviews_sample))
print("Blank reviews:", (reviews_sample["review"].str.strip() == "").sum())
print("Duplicate rows:", reviews_sample.duplicated().sum())
print()
print("Review length in characters:")
print(reviews_sample["review"].str.len().describe().round(1))

### 💡 What Sarah noticed

- **Blank reviews exist.** Roughly 1–2% in the real file. She decided to drop them — no text, no signal.
- **Duplicates exist** (a copy-paste bug in the export). She deduplicated before scoring.
- **Very short reviews ("ok", "fine")** are ambiguous. She flagged these separately; they aren't really positive or negative.
- **Review lengths vary widely** — from 1 word to 500+. No transformation needed for TextBlob, but she kept this in mind for Step 5.

**Cleaning isn't glamorous, but it's where hours of a real project go.** Every minute here saves ten minutes of confused results later.

## Step 4 — Train the model

Sarah did **not** train a new model. She used **TextBlob**, a library with a pre-trained sentiment model someone else built on a large body of English text.

This is legitimate. In the real world:

- For a **common task** (sentiment, topic tagging, language detection, face detection), there is usually a good pre-trained model you can reuse. **Start there.**
- For a **business-specific task** (which of *our* customers will churn, what does a *NorthStar* return reason look like), you must train your own. We'll do that in L03 and L04.

**Why is this step so short for Sarah?** Because 80% of the work in any ML project happens *around* the model, not in it. The framing, the data, the evaluation, the rollout — those are where the time goes.

## Step 5 — Evaluate on unseen data

Sarah cannot check all 10,000 predictions by hand. But she *can* spot-check a small sample to see whether the model is trustworthy on NorthStar's kind of text.

She hand-labels 15 reviews herself and compares.

In [ ]:
# A hand-labelled spot-check sample.
# Sarah reads each review and writes down her own verdict first,
# then scores them with TextBlob and compares.
spot_check = pd.DataFrame({
    "review": [
        "Absolutely love this dress, fits perfectly.",
        "Runs way too small, had to return.",
        "Arrived late but the quality is fine.",
        "Terrible customer service, very rude.",
        "Good value for money, happy with purchase.",
        "Poor quality fabric fell apart after one wash.",
        "Exactly as described, fast shipping, five stars.",
        "Not what I expected at all, looks nothing like the photo.",
        "Fits great but the colour faded quickly.",
        "Perfect gift, she loved it.",
        "Size chart is misleading, ordered medium got small.",
        "Comfortable and cute, would buy again.",
        "Shipping box was damaged, product okay.",
        "Beautiful design, high quality, lovely packaging.",
        "Never received the item, still waiting for refund.",
    ],
    "sarah_label": [
        "positive", "negative", "negative", "negative", "positive",
        "negative", "positive", "negative", "negative", "positive",
        "negative", "positive", "negative", "positive", "negative",
    ],
})

# Score each review with TextBlob and convert polarity to positive/negative.
def textblob_label(text):
    polarity = TextBlob(text).sentiment.polarity
    return "positive" if polarity >= 0 else "negative"

spot_check["model_label"] = spot_check["review"].apply(textblob_label)
spot_check["match"] = (spot_check["sarah_label"] == spot_check["model_label"])

accuracy = spot_check["match"].mean()
print(f"Model agreed with Sarah on {spot_check['match'].sum()} out of {len(spot_check)} reviews  "
      f"→ accuracy ≈ {accuracy:.0%}")
print()
print("Reviews where the model disagreed with Sarah:")
print(spot_check.loc[~spot_check["match"], ["review", "sarah_label", "model_label"]].to_string(index=False))

### 💡 What this evaluation tells Sarah

- **The accuracy number is only as good as the labels she wrote.** Another person might disagree with Sarah on some reviews — that's normal.
- **The disagreements are the most interesting row.** They usually point to *mixed* reviews ("fits great but colour faded") — the model tends to pick the dominant mood, which may or may not be what Aisha cares about.
- **15 reviews is a small sample.** Sarah should not claim a single accuracy percentage in her Friday deck; instead she'll say *"we spot-checked ~15 reviews and the model agreed on most, with mixed reviews being the main failure mode."*
- **We'll formalise this in L03** (precision, recall, confusion matrix). For now, just notice that *evaluation is about trust*, not a number.

**Accuracy alone is never enough.** Always look at *where* the model is wrong.

## Steps 6 & 7 — Deploy and Monitor (the hidden 40%)

Sarah's model runs in a notebook. That's fine for one report. It is **not** fine for a weekly process.

Typical deployment shapes:

- A **batch pipeline** that runs every Monday, scores new reviews, writes a CSV, and emails it to Aisha. (Probably what Sarah starts with next sprint.)
- A **dashboard** in NorthStar's BI tool where Aisha filters reviews by predicted theme.
- A **real-time API** a customer-service tool calls when an agent opens a review.

Once deployed, **monitoring** is essential:

- **Performance:** are last week's predictions still matching human spot-checks?
- **Drift:** have review lengths, vocabulary, or topic mixes shifted since the model was trained?
- **Business outcome:** does the team actually use the output? Are the top-flagged themes being acted on?

**When any of these degrade, it's time to revisit the model.**

## 🧠 Apply the workflow to a new scenario

**A hospital wants to predict which admitted patients are at high risk of readmission within 30 days.**

For each of the 7 steps, write **one concrete action Sarah would take** for this project. Take 5 minutes.

**Example for step 1:** *"Meet with the readmissions team to understand what counts as 'high-risk' — top 10% of all admissions? Or anyone above a specific probability threshold? Who decides?"*

*Your answers:*

1. Frame:
2. Collect:
3. Clean/explore:
4. Train:
5. Evaluate:
6. Deploy:
7. Monitor:

### Sample answer

1. **Frame:** Meet with clinicians and operations. Define "high-risk" clearly. Who acts on the prediction — the discharge nurse? A care coordinator? What intervention do they take? Success metric: reduction in 30-day readmissions among the flagged group.
2. **Collect:** Pull 2+ years of admissions data with features known at discharge (age, comorbidities, length of stay, lab results, medications, discharge destination) and labels (did they come back within 30 days?).
3. **Clean/explore:** Check for missing values, duplicate admissions, coding inconsistencies. Look at readmission rates by ward, age bracket, diagnosis — form hypotheses before modelling.
4. **Train:** Start with logistic regression as a baseline (L03). Try a random forest (L04) for comparison. Unlike Sarah's sentiment task, there is no generic pre-trained model — you must train your own.
5. **Evaluate:** Use a test set held back by time (train on 2022–2023, test on first half of 2024). Check recall for high-risk cases specifically, not just overall accuracy.
6. **Deploy:** Batch-score every patient at discharge, write the result to the electronic health record as a flag the coordinator sees.
7. **Monitor:** Weekly tracking of model accuracy and input distributions. Retrain quarterly or whenever drift is detected.

## ✅ Section 3 Summary

| Step | What happens | Where beginners get stuck |
|---|---|---|
| **1. Frame** | Business question, success metric, cost of wrongness | Skipping it entirely |
| **2. Collect** | Gather features (and labels, if training) | Assuming the data exists when it doesn't |
| **3. Clean/explore** | Fix missing, remove duplicates, check for bias | Underestimating how long this takes |
| **4. Train** | Fit a model — or reuse a pre-trained one | Training from scratch when a good pre-trained model exists |
| **5. Evaluate** | Spot-check or hold-out test | Reporting one number instead of *where* it's wrong |
| **6. Deploy** | Put the model where it gets used | Leaving the model stuck in a notebook |
| **7. Monitor** | Track performance over time | Assuming "trained once = good forever" |

**Key insight:**
> A beautifully trained model that solves the wrong problem is worth zero. The framing step (#1) is where senior practitioners earn their keep.

## 🏁 Module L01 complete — and the question Priya hasn't let go of

You've covered the three foundational sections:

- **Part 1:** What ML *is* — rules vs examples
- **Part 2:** The three categories — supervised, unsupervised, reinforcement
- **Part 3:** The 7-step workflow, from business question to deployed model

Sarah finalises her deck Thursday evening and walks into Priya's office Friday morning with the numbers. Priya nods, then looks up:

> *"Sarah — the numbers look great. But **how sure are we**? The model put a label on every review. How do we know it isn't confidently wrong in ways we can't see from this deck?"*

Sarah doesn't have a clean answer yet. That question — *how sure are we?* — is the engine of **Lesson 2**. Come to L02 ready to help Sarah answer it.

---

**Next step → `notebooks/assignment.ipynb`.**
Apply the full ML workflow to a completely different domain: Tom Bradley at Lakeside Bank is trying to predict loan default. Work through the practice tiers and the assignment exercises before our next session.